# 4. Report: every table, from saved results only (CPU, no dataset download)

Reads the per-block result files of the run from persistent storage and builds every table: depth error by
alignment protocol, by motion, distance, semantic class and confidence, paired comparisons, camera pose, scene
conditions (with the number of recordings behind each row), model against model, and the worst scenes. It downloads
nothing, runs in about a minute and can be re-run whenever another block has been scored.

The development block (val 11), on which every rule was tuned, is left out of all tables. Once the test split has
been scored (`05` to `07`), its scenes are part of these tables too; `07_test_report` shows them on their own.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
MODELS = ["vggt_omega_512", "vggt_1b"]
DEVELOPMENT_BLOCKS = [("val", 11)]    # every rule was tuned on this block, so it is left out of every table

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Load the run and build the tables ---
import pandas as pd
from vggt_aura import evaluation as ev, metrics as mt, pipeline as pl

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)
runs = {}
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    if len(scenes):
        development = scenes[[(s, b) in DEVELOPMENT_BLOCKS for s, b in zip(scenes['split'], scenes['block'])]]['scene_id']
        print(f"{model}: {scenes['scene_id'].nunique()} scenes loaded, {len(development)} of them development scenes left out of the tables")
        rows, scenes = rows[~rows['scene_id'].isin(development)], scenes[~scenes['scene_id'].isin(development)]
    runs[model] = (rows.reset_index(drop=True), scenes.reset_index(drop=True))
out_dir = session.persist_root / "metrics" / RUN_TAG / "report"
out_dir.mkdir(parents=True, exist_ok=True)


def show(title, table, digits=4):
    print()
    print("=====", title, "=====")
    print(table.round(digits).to_string(index=False) if len(table) else "(no rows)")


for model, (rows, scenes) in runs.items():
    if rows.empty:
        print(model, ": no results yet")
        continue
    print()
    print("#" * 30, model, "|", scenes["scene_id"].nunique(), "scenes |", scenes[["split", "block"]].drop_duplicates().shape[0], "blocks")
    report = ev.build_report(rows, scenes)
    for name, table in report.items():
        show(name, table)
        table.to_csv(out_dir / f"{model}_{name}.csv", index=False)

vggt_omega_512: 307 scenes loaded, 20 of them development scenes left out of the tables
vggt_1b: 307 scenes loaded, 20 of them development scenes left out of the tables

############################## vggt_omega_512 | 287 scenes | 15 blocks

===== headline_by_protocol =====
         protocol  n_scenes    pixels  abs_rel  ci_low  ci_high  thin  delta125
      frame_scale       287 452290446   0.1047  0.0933   0.1188 False    0.9187
frame_scale_shift       287 452290446   0.1030  0.0931   0.1151 False    0.9150
       pose_scale       281 441926197   0.4300  0.2120   0.7852 False    0.7917
   sequence_scale       287 452290447   0.1134  0.1018   0.1276 False    0.9135
        unaligned       287 452290447   0.9835  0.9814   0.9854 False    0.0000

===== motion =====
         stratum  n_scenes    pixels  abs_rel  ci_low  ci_high  thin  delta125
      background       287 423729775   0.1044  0.0938   0.1186 False    0.9171
   moving object       270  13595792   0.2661  0.2378   0.2968 Fals

In [5]:
# --- 5. Model against model, paired by scene ---
if all(not runs[m][0].empty for m in MODELS) and len(MODELS) == 2:
    a, b = MODELS
    comparison = ev.compare_models(runs[a][0], runs[a][1], runs[b][0], runs[b][1], a, b)
    show(f"{a} minus {b}, paired by scene  [{mt.PRIMARY_PROTOCOL}]", comparison)
    comparison.to_csv(out_dir / "model_comparison.csv", index=False)
else:
    print("need results for exactly two models to compare")


===== vggt_omega_512 minus vggt_1b, paired by scene  [sequence_scale] =====
                   quantity  n_scenes  vggt_omega_512  vggt_1b  difference_a_minus_b   ci_low  ci_high  scenes_where_a_is_lower
         AbsRel, all pixels       287          0.1134   0.1204               -0.0070  -0.0145   0.0018                      211
         AbsRel, background       287          0.1044   0.1130               -0.0086  -0.0157   0.0003                      215
      AbsRel, parked object       199          0.1724   0.1942               -0.0218  -0.0428   0.0018                      138
      AbsRel, moving object       270          0.2661   0.2921               -0.0260  -0.0511  -0.0006                      164
        rotation_deg_median       269          1.2155   2.8268               -1.6114  -2.1354  -1.1228                      230
     translation_deg_median       269          3.8010  10.6554               -6.8543 -11.1648  -3.2236                      178
                      auc30

In [6]:
# --- 6. One line per scene, and the scenes where each model does worst ---
# "recording" is the drive a scene was cut from: many bad scenes from ONE recording are one problem, not many.
# scale_disagreement: 0 = pose and depth agree on the scale, 0.69 = they differ by a factor of two.
KEEP = ["scene_id", "block", "road_type", "weather_group", "lighting", "speed_kph_median", "abs_rel", "abs_rel_moving",
        "rotation_deg_median", "translation_deg_median", "ate_scale_only_pct_of_path", "pose_scale_over_depth_scale"]
for model, (rows, scenes) in runs.items():
    if rows.empty:
        continue
    overview = ev.scene_overview(rows, scenes)
    overview.to_csv(out_dir / f"{model}_scene_overview.csv", index=False)
    print()
    print("#" * 30, model)
    for title, by, moving_only in (("worst depth (AbsRel, all pixels)", "abs_rel", False),
                                   ("worst translation direction, moving scenes only", "translation_deg_median", True),
                                   ("pose scale and depth scale disagree most, moving scenes only", "scale_disagreement", True)):
        worst = ev.worst_scenes(overview, by, n=12, moving_only=moving_only)
        show(f"{title} | {worst['recording'].nunique()} recording(s) among these {len(worst)}",
             worst[[c for c in KEEP if c in worst]], digits=3)


############################## vggt_omega_512

===== worst depth (AbsRel, all pixels) | 2 recording(s) among these 12 =====
               scene_id  block road_type weather_group lighting  speed_kph_median  abs_rel  abs_rel_moving  rotation_deg_median  translation_deg_median  ate_scale_only_pct_of_path  pose_scale_over_depth_scale
 2026-01-08-15-27-06|43     10  overland           wet twilight             63.68    1.640           0.499                0.323                   0.653                       1.505                        1.116
 2026-01-08-15-27-06|42      2  overland           wet twilight             45.70    0.619           1.114                0.262                   0.563                       2.311                        1.224
 2026-01-08-15-27-06|25     11     urban           wet twilight             10.27    0.387           0.498                0.189                   0.730                       0.635                        1.046
 2026-01-08-16-15-15|24     13  overlan